# Synthetic Eval Workbench

Run a fresh synthetic-persona benchmark into `outputs/`, then inspect summary metrics, benchmark integrity, persona errors, and item-level bias tables. The notebook reads the supported minimal config surface from `.env`.

In [5]:
from pathlib import Path
import os
import sys

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

repo_root = Path.cwd()
while not (repo_root / "app").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

load_dotenv(repo_root / ".env", override=True)

from app.notebook_eval import (
    build_cluster_collapse_table,
    build_item_error_table,
    build_opening_signal_table,
    build_persona_error_table,
    build_runtime_anomaly_table,
    build_runtime_support_gap_table,
    load_benchmark_integrity,
    load_eval_metrics,
    load_eval_records,
    run_eval_notebook,
    split_item_error_table,
)


In [6]:
# Primary run controls. Backend/model and benchmark defaults come from .env.
personas = 4
seed = 42
max_api_calls = 500
save_diagnostics = False
debug_outputs = True
trace_level = "off"
output_dir = repo_root / "outputs"
run_eval_now = True


In [7]:
detector_backend = os.getenv("DETECTOR_BACKEND", "openrouter").strip().lower() or "openrouter"
if detector_backend == "ollama":
    detector_target = os.getenv("OLLAMA_DETECTOR_MODEL", "qwen3.5:4b")
else:
    detector_target = os.getenv("OPENROUTER_DETECTOR_MODEL", "openrouter/auto")

print(f"Configured detector backend: {detector_backend} [{detector_target}]")
print("Persona runtime: deterministic simulator")


Configured detector backend: openrouter [openai/gpt-oss-120b]
Persona runtime: deterministic simulator


In [8]:
if run_eval_now:
    run_summary = run_eval_notebook(
        persona_count=personas,
        seed=seed,
        save_diagnostics=save_diagnostics,
        max_api_calls=max_api_calls,
        trace_level=trace_level,
        debug_outputs=debug_outputs,
        output_dir=output_dir,
    )
    output_dir = Path(run_summary["output_dir"])
else:
    run_summary = {"output_dir": str(Path(output_dir).resolve())}
    output_dir = Path(run_summary["output_dir"])

print(f"Artifacts: {output_dir}")


Running synthetic eval: personas=4, live_status=on
Stop policy: MIN_TURNS=20 | MAX_TURNS=40 | STOP_CONFIDENCE=0.66
Runtime controls: DIAGNOSIS_AGENT_USE_LLM=0 | DETERMINISTIC_BDI_LABEL_THRESHOLD=14 | DETECTOR_EXTRACTOR_MAX_NEW_TOKENS=800
[eval 1/4 persona=1] cycle=21 turn=19 stage=detector_graph conf=81.7% calls=57/500
[eval 2/4 persona=2] cycle=21 turn=19 stage=detector_graph conf=79.8% calls=114/500
[eval 3/4 persona=3] cycle=21 turn=19 stage=detector_graph conf=77.2% calls=176/500
[eval 4/4 persona=4] cycle=21 turn=19 stage=detector_graph conf=92.9% calls=225/500
item_f1=0.1238 objective=0.0488
Artifacts: /home/mdel2424/dev/eRisk_Honours/outputs


In [9]:
metrics = load_eval_metrics(output_dir)
benchmark_integrity = load_benchmark_integrity(output_dir)
records_df = load_eval_records(output_dir)
persona_error_df = build_persona_error_table(records_df)
item_error_df = build_item_error_table(records_df)
item_views = split_item_error_table(item_error_df)


In [10]:
def style_table(df: pd.DataFrame):
    if df.empty:
        return df
    format_map = {
        col: "{:.3f}"
        for col in ["avg_pred", "avg_true", "mean_error", "abs_mean_error", "bdi_error", "bdi_abs_error"]
        if col in df.columns
    }
    if "mean_error" in df.columns:
        return df.style.format(format_map).background_gradient(subset=["mean_error"], cmap="RdYlGn", vmin=-1.5, vmax=1.5)
    return df.style.format(format_map)


In [11]:
summary_df = pd.DataFrame([
    {
        "evaluation_mode": metrics.get("evaluation_mode", "synthetic"),
        "persona_count": metrics.get("persona_count", 0),
        "item_f1_macro_at_1": metrics.get("item_f1_macro_at_1", 0.0),
        "item_mae": metrics.get("item_mae", 0.0),
        "bdi_mae": metrics.get("bdi_mae", 0.0),
        "avg_turns_to_decision": metrics.get("avg_turns_to_decision", 0.0),
        "objective": metrics.get("objective", 0.0),
    }
])
display(summary_df)

integrity_df = pd.DataFrame([
    {
        "integrity_pass": benchmark_integrity.get("pass", False),
        "detector_backend": benchmark_integrity.get("detector", {}).get("backend", ""),
        "detector_target": benchmark_integrity.get("detector", {}).get("target", ""),
        "prior_manifest_exists": benchmark_integrity.get("prior_manifest", {}).get("exists", False),
        "prior_manifest_matches_current": benchmark_integrity.get("prior_manifest", {}).get("matches_current", None),
        "results_alignment_pass": benchmark_integrity.get("results_alignment", {}).get("pass", False),
        "manifest_consistency_pass": benchmark_integrity.get("manifest_consistency", {}).get("pass", False),
    }
])
display(integrity_df)

if benchmark_integrity.get("issues"):
    print("Integrity issues:", benchmark_integrity["issues"])

runtime_anomaly_df = build_runtime_anomaly_table(records_df)
print(f"Runtime anomaly rows: {len(runtime_anomaly_df)}")
if not runtime_anomaly_df.empty:
    display(runtime_anomaly_df)

cluster_collapse_df = build_cluster_collapse_table(records_df)
print(f"Cluster collapse rows: {len(cluster_collapse_df)}")
if not cluster_collapse_df.empty:
    display(cluster_collapse_df)

support_gap_df = build_runtime_support_gap_table(records_df)
avg_supported = records_df["runtime_supported_item_count"].mean() if not records_df.empty else 0.0
print(f"Average supported item count: {avg_supported:.2f}")
print(f"Low-support high-error rows: {len(support_gap_df)}")
if not support_gap_df.empty:
    display(support_gap_df)

opening_signal_df = build_opening_signal_table(records_df)
print(f"Opening bootstrap rows: {len(opening_signal_df)}")
if not opening_signal_df.empty:
    display(opening_signal_df)

if not records_df.empty and "runtime_opening_bootstrap_applied" in records_df.columns:
    supported_split = records_df.groupby(
        records_df["runtime_opening_bootstrap_applied"].fillna(False).astype(bool)
    )["runtime_supported_item_count"].mean()
    print("Average supported item count by bootstrap success:")
    display(supported_split.rename(index={True: "bootstrap_applied", False: "no_bootstrap"}).to_frame("avg_supported_item_count"))

if not opening_signal_df.empty:
    cognitive_to_somatic = opening_signal_df.loc[
        (opening_signal_df["runtime_opening_bootstrap_cluster"] == "cognitive_affective")
        & (opening_signal_df["runtime_opening_followup_cluster"] == "somatic_vegetative")
    ]
    print(f"Cognitive-opening personas that pivoted somatic first: {len(cognitive_to_somatic)}")
    if not cognitive_to_somatic.empty:
        display(cognitive_to_somatic)


,evaluation_mode,persona_count,item_f1_macro_at_1,item_mae,bdi_mae,avg_turns_to_decision,objective
0,synthetic,4,0.1238,0.9405,19.25,20.0,0.0488


,integrity_pass,detector_backend,detector_target,prior_manifest_exists,prior_manifest_matches_current,results_alignment_pass,manifest_consistency_pass
0,True,openrouter,openai/gpt-oss-120b,True,False,True,True


Runtime anomaly rows: 3


,persona_id,family,source,bdi_true,bdi_pred,runtime_active_cluster,runtime_evidence_binding_coverage,runtime_diagnosis_confidence,runtime_supported_item_count,runtime_somatic_posterior,runtime_cognitive_posterior,predicted_nonzero_item_count,predicted_somatic_item_count,predicted_cognitive_item_count,dominant_predicted_cluster,somatic_cluster_share,bdi_abs_error
0,2,somatic_evasive,synthetic,32,2,somatic_vegetative,1.0,0.596749,2,0.6215,0.1800,2,2,0,somatic_vegetative,1.0,30
1,1,risk_leaning,synthetic,28,3,somatic_vegetative,1.0,0.650631,3,0.6111,0.2042,3,3,0,somatic_vegetative,1.0,25
2,3,cognitive_ruminative,synthetic,21,3,somatic_vegetative,1.0,0.662584,3,0.5896,0.1800,3,3,0,somatic_vegetative,1.0,18


Cluster collapse rows: 3


,persona_id,family,source,bdi_true,bdi_pred,runtime_active_cluster,runtime_evidence_binding_coverage,runtime_diagnosis_confidence,runtime_supported_item_count,runtime_somatic_posterior,runtime_cognitive_posterior,predicted_nonzero_item_count,predicted_somatic_item_count,predicted_cognitive_item_count,dominant_predicted_cluster,somatic_cluster_share,bdi_abs_error
0,2,somatic_evasive,synthetic,32,2,somatic_vegetative,1.0,0.596749,2,0.6215,0.1800,2,2,0,somatic_vegetative,1.0,30
1,1,risk_leaning,synthetic,28,3,somatic_vegetative,1.0,0.650631,3,0.6111,0.2042,3,3,0,somatic_vegetative,1.0,25
2,3,cognitive_ruminative,synthetic,21,3,somatic_vegetative,1.0,0.662584,3,0.5896,0.1800,3,3,0,somatic_vegetative,1.0,18


Average supported item count: 2.00
Low-support high-error rows: 1


,persona_id,family,source,bdi_true,bdi_pred,bdi_abs_error,runtime_supported_item_count,runtime_diagnosis_confidence,runtime_active_cluster
0,2,somatic_evasive,synthetic,32,2,30,2,0.596749,somatic_vegetative


In [12]:
display(style_table(persona_error_df.head(20)))


,persona_id,split,family,source,bdi_true,bdi_pred,bdi_error,bdi_abs_error
0,2,eval,somatic_evasive,synthetic,32,2,-30.000,30.000
1,1,eval,risk_leaning,synthetic,28,3,-25.000,25.000
2,3,eval,cognitive_ruminative,synthetic,21,3,-18.000,18.000
3,4,eval,cognitive_ruminative,synthetic,4,0,-4.000,4.000


In [13]:
display(style_table(item_views["all_items"].sort_values(["mean_error", "item_id"]).reset_index(drop=True)))


,item_id,symptom_name,avg_pred,avg_true,mean_error,abs_mean_error,n_profiles
0,7,Self-Dislike,0.000,1.750,-1.750,1.750,4
1,14,Worthlessness,0.000,1.750,-1.750,1.750,4
2,5,Guilty Feelings,0.000,1.250,-1.250,1.250,4
3,8,Self-Criticalness,0.000,1.250,-1.250,1.250,4
4,11,Agitation,0.000,1.250,-1.250,1.250,4
5,19,Concentration Difficulty,0.000,1.250,-1.250,1.250,4
6,2,Pessimism,0.000,1.000,-1.000,1.000,4
7,4,Loss of Pleasure,0.000,1.000,-1.000,1.000,4
8,6,Punishment Feelings,0.000,1.000,-1.000,1.000,4
9,12,Loss of Interest,0.000,1.000,-1.000,1.000,4


In [14]:
display(style_table(item_views["under_predicted"].reset_index(drop=True)))


,item_id,symptom_name,avg_pred,avg_true,mean_error,abs_mean_error,n_profiles
0,7,Self-Dislike,0.000,1.750,-1.750,1.750,4
1,14,Worthlessness,0.000,1.750,-1.750,1.750,4
2,5,Guilty Feelings,0.000,1.250,-1.250,1.250,4
3,8,Self-Criticalness,0.000,1.250,-1.250,1.250,4
4,11,Agitation,0.000,1.250,-1.250,1.250,4
5,19,Concentration Difficulty,0.000,1.250,-1.250,1.250,4
6,2,Pessimism,0.000,1.000,-1.000,1.000,4
7,4,Loss of Pleasure,0.000,1.000,-1.000,1.000,4
8,6,Punishment Feelings,0.000,1.000,-1.000,1.000,4
9,12,Loss of Interest,0.000,1.000,-1.000,1.000,4


In [15]:
display(style_table(item_views["over_predicted"].reset_index(drop=True)))


,item_id,symptom_name,avg_pred,avg_true,mean_error,abs_mean_error,n_profiles
